# 집계 후보 합집합: 다른 영상 점검 (API 중단 시점)

`0936`에서 후보 합집합으로 7회 확정·1회 미확정이 나와, `0188·0518·0642`에서도 후보 누락과 최종 집계 단계를 나누어 확인했습니다. 기존 0.5초 실행과 같은 질문 해석을 고정했습니다. `0642`의 새 0.25초 **전체 탐색 응답은 저장됐지만**, 후보 검증 중 OpenAI API가 `429 credit_balance_exhausted`를 반환해 추가 호출을 중단했습니다. 따라서 아래는 **완료된 후보 합집합 성능 평가가 아닙니다.**

## 현재까지 확인한 수치

| 영상 | 사람 기준 | 기존 0.5초 후보 제안 | 기존 후보가 있는 사건 | 기존 최종 확정 | 새 0.25초 탐색 |
|---|---:|---:|---:|---:|---|
| `0188` 울타리 | 사용자 2 | 10 | 2/2 | 2 | 미실행 |
| `0518` 소파 | 잠정 2 | 6 | 2/2 | 2 | 미실행 |
| `0642` 분수 | 잠정 2 | 4 | 2/2 | 1 | **탐색만 완료: 후보 8개, 시각 매칭 2/2** |

후보와 사람 기준 사건의 매칭은 **검토 구간 ±0.75초 안의 시각**을 일대일로 맞춘 것입니다. 후보가 행동 진입을 실제로 증명한다는 뜻은 아닙니다. `0518·0642`의 사건 시각도 사람의 독립 확정 라벨이 아닌 프레임 잠정 검토입니다. `0188`은 사용자가 두 구간을 직접 확인했습니다.

In [1]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks": root = root.parent
report_path = root / "notebooks/smarthome-bench-evidence/new-pipeline/ablation-cross-video/offline-audit.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
for case in report["cases"]:
    print(case["video_id"], "기존 후보", case["step050_scan"]["proposal_count"],
          "사건 시각 매칭", f'{case["step050_scan"]["reference_events_with_candidate"]}/{case["reference_count"]}',
          "확정", case["step050_confirmed"], "새 탐색", case["step025_status"])

smartbench_0188 기존 후보 10 사건 시각 매칭 2/2 확정 2 새 탐색 not_run_api_quota_exhausted
smartbench_0518 기존 후보 6 사건 시각 매칭 2/2 확정 2 새 탐색 not_run_api_quota_exhausted
smartbench_0642 기존 후보 4 사건 시각 매칭 2/2 확정 1 새 탐색 scan_only_api_quota_exhausted

## `0642`: 탐색 간격보다 후보 시각과 검증 구간이 문제

기존 0.5초 실행의 두 번째 입수 후보는 **6.25초**, 새 0.25초 탐색의 대응 후보는 **6.5초**입니다. 잠정 사람 기준 두 번째 입수 구간은 4~6초입니다. 기존 후보를 검증할 때 사용한 첫 근거 프레임은 5.0초였는데, 모델의 독립 검사는 **그때 이미 강아지가 물에 부분적으로 들어가 있었다**고 응답해 ‘물 밖 → 물 안’ 전환을 확정하지 못했습니다. 즉, 후보 시각이 실제 진입보다 늦으면 1.25초 앞에서 시작하는 짧은 검증 창도 이전 상태를 놓칩니다.

새 탐색은 8.0·12.0·12.25·13.0·13.75·14.5초에도 후보를 냈습니다. 저장된 응답 중 13.0·14.5초는 스스로 **계속 물속에 머무는 장면**이라고 설명합니다. 이 후보들을 새 입수로 세면 중복 집계가 됩니다. 새 후보의 전·후 검증은 API 중단으로 끝내지 못했습니다.

In [ ]:
case = next(item for item in report["cases"] if item["video_id"] == "smartbench_0642")
print("기존 후보 시각:", case["step050_scan"]["candidate_times"])
print("새 탐색 후보 시각:", case["step025_scan"]["candidate_times"])
print("새 탐색 상태:", case["step025_status"])

## 원본 영상 확인

로컬 `data/videos/`에 파일이 있으면 아래 셀로 영상을 재생할 수 있습니다. `0642`에서는 약 4~7초에 두 번째 입수 전·후 장면이 보이는지 확인해 주세요. 저장된 표와 JSON을 읽는 데에는 영상이나 API 키가 필요 없습니다.

In [ ]:
for video_id in ("0188", "0518", "0642"):
    video = root / "data/videos" / f"smartbench_{video_id}.mp4"
    print(video_id, video if video.exists() else "로컬 영상 없음")
    if video.exists():
        try:
            from IPython.display import Video, display
        except ImportError:
            print("Jupyter/IPython에서 재생 가능")
        else:
            display(Video(filename=str(video), embed=False, html_attributes="controls"))

## 다음 검증

API 사용이 다시 가능해지면 `0642`의 저장된 0.25초 탐색부터 후보 검증을 이어갑니다. 4~6초의 실제 진입 전 장면을 포함하도록 검증 창을 넓혀 **같은 강아지인지, 물 밖에서 물 안으로 들어갔는지**를 먼저 확인해야 합니다. `0188·0518`은 기존 탐색에서 알려진 사건을 이미 모두 찾아 확정했으므로, 새 촘촘한 탐색의 우선순위는 낮습니다. 후보 합집합이 다른 영상에서도 중복 없이 동작한다는 결론은 아직 낼 수 없습니다.